[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/55_rope_2d_image_tokens_solution.ipynb)

# 🔴 Solution: 2D RoPE for Image Tokens

Reference solution for `rope_2d_image_tokens`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math


In [ ]:
# ✅ SOLUTION

def _rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def _rope_1d(x: torch.Tensor, positions: torch.Tensor, base: float) -> torch.Tensor:
    dim = x.shape[-1]
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=x.device, dtype=x.dtype) / dim))
    angles = positions.to(device=x.device, dtype=x.dtype).unsqueeze(-1) * inv_freq
    cos = torch.repeat_interleave(torch.cos(angles), 2, dim=-1).unsqueeze(0).unsqueeze(0)
    sin = torch.repeat_interleave(torch.sin(angles), 2, dim=-1).unsqueeze(0).unsqueeze(0)
    return x * cos + _rotate_half(x) * sin


def apply_2d_rope(x: torch.Tensor, height: int, width: int, base: float = 10000.0) -> torch.Tensor:
    B, Hh, N, D = x.shape
    if N != height * width:
        raise ValueError("sequence length must equal height * width")
    if D % 4 != 0:
        raise ValueError("head_dim must be divisible by 4")
    half = D // 2
    y_pos = torch.arange(height, device=x.device).repeat_interleave(width)
    x_pos = torch.arange(width, device=x.device).repeat(height)
    y_part = _rope_1d(x[..., :half], y_pos, base)
    x_part = _rope_1d(x[..., half:], x_pos, base)
    return torch.cat([y_part, x_part], dim=-1)


In [ ]:
# Verify
x = torch.randn(2, 4, 3 * 3, 8)
y = apply_2d_rope(x, height=3, width=3)
print(y.shape)
print("norm preserved:", torch.allclose(y.norm(dim=-1), x.norm(dim=-1), atol=1e-5))


In [ ]:
# Run judge
from torch_judge import check
check('rope_2d_image_tokens')
